# 08 FS2 Benchmark And Shortlisting

This notebook is the first fair shortlisting checkpoint.

What it does conceptually:
- collect the active saved benchmark runs
- compare the `FS2` contenders side by side
- confirm that no shortlist is taken after `FS1`
- prepare the survivor set for later `FS3` work

What it does **not** do now:
- rerun the benchmarks
- regenerate heavy outputs
- execute the shortlist automatically


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


In [ ]:
rows = []
for run_label in ['naive_benchmark', 'lear_benchmark', 'xgboost_benchmark', 'prophet_benchmark', 'model_comparison']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))


In [ ]:
display(shortlisting_policy_frame())


In [ ]:
run_dir = latest_run_or_none("model_comparison")

if run_dir is None:
    print("No combined model comparison run exists yet under the finalized methodology.")
else:
    print(run_dir)
    metrics_by_reporting_level = load_csv(run_dir, "metrics_by_reporting_level.csv")
    official_naive = load_json(run_dir, "official_naive_reference.json")
    focus_models = [
        "naive_previous_day",
        "naive_previous_week",
        "naive_previous_year",
        "lear_fs2",
        "xgboost_fs2",
        "prophet_fs2",
    ]
    display(pd.DataFrame([official_naive]))
    display(
        metrics_by_reporting_level[metrics_by_reporting_level["model"].isin(focus_models)]
        .sort_values(["dataset_split", "reporting_level_sort_order", "mae", "model"])
        .reset_index(drop=True)
    )


## Optional execution hook


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:
    estimate = estimate_run_duration_seconds(output_root, "model_comparison")
    if estimate is not None:
        print(
            "Heavy rerun warning: latest comparable run "
            f"{estimate['run_id']} suggests about {format_duration(float(estimate['estimate_seconds']))}."
        )
    else:
        print("Heavy rerun warning: no comparable runtime estimate was found for this stage.")

    command = [sys.executable, str(PACKAGE_ROOT / "run_model_comparison.py")]

    started = time.perf_counter()
    subprocess.run(command, check=True)
    elapsed_seconds = time.perf_counter() - started
    print(f"Actual wall-clock time: {format_duration(elapsed_seconds)}")
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True only when you are ready to execute the finalized pipeline.")
